In [ ]:
# Mock Data Generator (Smart Conveyor Monitoring System)
Simulates FB1 tag shape (Motor_Run, SystemReady, FaultActive, ItemCount,
TempAlarmHigh, TempAlarmLow) using a random-walk temperature model.
OPC-UA live feed is architecturally blocked (see project README) —
this generates realistic mock data so CSV logging / alarm detection /
dashboard can be built and tested now.
## CSV Logging
Appends readings to a persistent log (real historian behavior — survives
kernel restarts, never auto-wiped). Uses csv.DictWriter for safe column
alignment.

In [17]:
import random

def initial_state():
    """Starting values for one mock conveyor. Call once per machine."""
    return{
    "temp": 22.0,
    "item_count": 0,
    "fault_active": False,
    }

In [18]:
def next_reading(state, fault_chance=0.02, reset_fault=False):
    """
    Advance one mock conveyor by a single time step.
    Returns a NEW state dict — does not mutate the input.
    """
    fault_active = state["fault_active"]
    if reset_fault:
        fault_active = False

    step = random.uniform(-0.5, 0.5)
    new_temp = state["temp"] + step
    new_temp = max(0.0, min(95.0, new_temp)) # keep in physically plausible range
    new_temp = round(new_temp, 2) #realistic sensor precision, not 15 decimals

    temp_alarm_high = new_temp > 80.0
    temp_alarm_low = new_temp < 5.0

    if not fault_active and random.random() < fault_chance:
        fault_active = True

    motor_run = not fault_active
    item_count = state["item_count"]
    if motor_run:
        item_count += 1
        if item_count > 9999:
            item_count = 0


    system_ready = (not fault_active) and (not temp_alarm_high) and (not temp_alarm_low)

    return{
    "temp" : new_temp,
    "item_count" : item_count,
    "fault_active" : fault_active,
    "motor_run" : motor_run,
    "system_ready" : system_ready,
    "temp_alarm_high" : temp_alarm_high,
    "temp_alarm_low" : temp_alarm_low,
    }

In [19]:
import csv
import os
from datetime import datetime

LOG_FILE = "conveyor_log.csv"
FIELDNAMES = ["timestamp", "temp", "item_count", "motor_run", "system_ready", "fault_active", "temp_alarm_high", "temp_alarm_low"]

def log_reading(state, log_file=LOG_FILE):
    """Append one conveyor reading to the persistent CSV log.
    Writes the header row only if the file doesn't exist yet.
    """
    file_exists = os.path.exists(log_file)
    with open(log_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()
        row = {"timestamp": datetime.now().isoformat(timespec="seconds")}
        row.update(state)
        writer.writerow(row)

In [20]:
state = initial_state()
for i in range(15):
    force_fault = 1.0 if i == 5 else 0.0
    reset = (i == 10)
    state = next_reading(state, fault_chance=force_fault, reset_fault=reset)
    log_reading(state)

# verify
with open(LOG_FILE) as f:
    print(f.read())

timestamp,temp,item_count,motor_run,system_ready,fault_active,temp_alarm_high,temp_alarm_low
2026-08-21T20:34:12,21.96,1,True,True,False,False,False
2026-08-21T20:34:12,22.32,2,True,True,False,False,False
2026-08-21T20:34:12,22.33,3,True,True,False,False,False
2026-08-21T20:34:12,22.02,4,True,True,False,False,False
2026-08-21T20:34:12,21.57,5,True,True,False,False,False
2026-08-21T20:34:12,21.78,5,False,False,True,False,False
2026-08-21T20:34:12,21.89,5,False,False,True,False,False
2026-08-21T20:34:12,22.09,5,False,False,True,False,False
2026-08-21T20:34:12,21.86,5,False,False,True,False,False
2026-08-21T20:34:12,21.47,5,False,False,True,False,False
2026-08-21T20:34:12,21.46,6,True,True,False,False,False
2026-08-21T20:34:12,21.26,7,True,True,False,False,False
2026-08-21T20:34:12,21.14,8,True,True,False,False,False
2026-08-21T20:34:12,21.49,9,True,True,False,False,False
2026-08-21T20:34:12,21.02,10,True,True,False,False,False



In [ ]:
## Summary — Mock Data Generator
- Built `next_reading()`: random-walk temp (not fresh-random) so alarm
  detection can be tested against continuous, physically plausible data.
- State passed as dict in/out (not global) — enables multi-machine sim later.
- Fault latches until explicit `reset_fault=True` — matches FB1's
  reset-dominant E-Stop behavior.
- Temp clamped [0, 95] — prevents unbounded random-walk drift outside
  any real conveyor's operating range.

In [ ]:
## Summary — CSV Logging
- Diagnosed "w" mode inside a loop truncating the file each write — only
  last row survived. Fixed by opening once (or using "a" append mode).
- Chose append mode ("a") over fresh-write ("w") to match real historian
  behavior: data persists across kernel restarts, never silently wiped.
- csv.DictWriter matches columns by field name, not position — avoids
  silent column-misalignment bugs.
- Fixed float precision: round(temp, 2) at the SOURCE (next_reading),
  not at the logging point — every consumer of state["temp"] benefits,
  not just the CSV.
- Next: wire into a continuous loop with time.sleep() for realistic
  timed readings (vs. instant 15-iteration burst).